# Day 31: 过拟合防护 + ML策略实战 + Auquan收尾

**日期**: 2026-08-09 | **阶段**: 补学 Day 5/8 | **时长**: 5小时

## 今日目标
- 亲手演示多项式过拟合 → 样本外验证 → 直观理解过拟合
- ML策略实战：用 sklearn 构建交易信号生成器
- Auquan 收尾总结

## Part 1: 多项式过拟合可视化

核心理解：用高次多项式完美拟合噪声数据 → 训练集满分 → 测试集崩盘。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'PingFang SC']
plt.rcParams['axes.unicode_minus'] = False

# 生成带噪声的真实数据
np.random.seed(42)
n = 50
X_all = np.linspace(0, 10, n).reshape(-1, 1)
# 真实关系：y = 2x + 3sin(x) + 噪声
y_true_func = lambda x: 2*x + 3*np.sin(x)
y_all = y_true_func(X_all.ravel()) + np.random.randn(n) * 2

# 前30个训练，后20个测试（时序划分！不是随机打乱）
X_train, X_test = X_all[:30], X_all[30:]
y_train, y_test = y_all[:30], y_all[30:]

# 画数据分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, X, y, title in [(axes[0], X_train, y_train, '训练集（前30个）'),
                          (axes[1], X_test, y_test, '测试集（后20个）')]:
    ax.scatter(X, y, alpha=0.7, s=40)
    ax.plot(X_all, y_true_func(X_all.ravel()), '--', color='gray', alpha=0.5, label='真实函数')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('X'); ax.set_ylabel('y')
    ax.legend()
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day31/data_split.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'训练集: {len(X_train)} 个点 | 测试集: {len(X_test)} 个点')
print('⚠️ 关键：时序数据按时间切分，不能随机打乱！')

## Part 2: 不同复杂度模型的过拟合对比

分别用 1次（线性）、3次、9次、15次多项式拟合，看训练集/测试集得分变化。

In [ ]:
degrees = [1, 3, 9, 15]
X_plot = np.linspace(0, 10, 200).reshape(-1, 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
results = []

for i, deg in enumerate(degrees):
    ax = axes[i // 2][i % 2]
    
    # 多项式回归
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_train, y_train)
    
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    y_pred = model.predict(X_plot)
    
    # 画图
    ax.scatter(X_train, y_train, alpha=0.5, s=30, label='训练集')
    ax.scatter(X_test, y_test, alpha=0.7, s=40, color='red', label='测试集')
    ax.plot(X_plot, y_pred, '--', lw=2, color='green', label=f'{deg}次拟合')
    ax.plot(X_all, y_true_func(X_all.ravel()), ':', color='gray', alpha=0.5, label='真实')
    ax.set_title(f'{deg}次多项式 | 训练R²={train_score:.3f} | 测试R²={test_score:.3f}', fontsize=13)
    ax.set_xlabel('X'); ax.set_ylabel('y')
    ax.legend(fontsize=8)
    ax.set_ylim(-10, 30)
    
    results.append({'degree': deg, 'train_r2': train_score, 'test_r2': test_score})

plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day31/overfitting_degrees.png', dpi=100, bbox_inches='tight')
plt.show()

# 总结表格
df = pd.DataFrame(results)
df['gap'] = df['train_r2'] - df['test_r2']
print(df.to_string(index=False))
print('\n💡 9次多项式的训练R²接近1（完美拟合噪声），但测试R²暴跌 → 这就是过拟合！')

## Part 3: 样本外验证 — 参数选择的过拟合陷阱

演示：用网格搜索在训练集上找「最优参数」，然后看它在测试集上的真实表现。

In [ ]:
from sklearn.metrics import mean_squared_error

# 用更多数据模拟真实场景
np.random.seed(123)
n_total = 200
X_all2 = np.linspace(0, 20, n_total).reshape(-1, 1)
y_all2 = 1.5 * X_all2.ravel() + 4 * np.sin(0.8 * X_all2.ravel()) + np.random.randn(n_total) * 3

# 时序划分：前70%训练，后30%测试
split = int(n_total * 0.7)
X_tr, X_te = X_all2[:split], X_all2[split:]
y_tr, y_te = y_all2[:split], y_all2[split:]

# 网格搜索：在训练集上尝试1到30次多项式
degrees_range = range(1, 31)
train_mse = []
test_mse = []

for deg in degrees_range:
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_tr, y_tr)
    train_mse.append(mean_squared_error(y_tr, model.predict(X_tr)))
    test_mse.append(mean_squared_error(y_te, model.predict(X_te)))

# 可视化：训练误差 vs 测试误差
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(degrees_range, train_mse, 'b-o', label='训练集 MSE', markersize=5)
ax1.plot(degrees_range, test_mse, 'r-o', label='测试集 MSE', markersize=5)
ax1.axvline(x=np.argmin(test_mse)+1, color='green', linestyle='--', label=f'最优={np.argmin(test_mse)+1}次')
ax1.set_xlabel('多项式次数'); ax1.set_ylabel('MSE')
ax1.set_title('训练误差 vs 测试误差（MSE）', fontsize=14)
ax1.legend()
ax1.set_yscale('log')

# 对比最优参数 vs 过拟合参数
best_deg = np.argmin(test_mse) + 1
worst_deg = np.argmin(train_mse) + 1  # 训练集上「最好」的

for deg, label, color in [(best_deg, f'{best_deg}次（泛化最优）', 'green'),
                            (worst_deg, f'{worst_deg}次（训练最优=过拟合）', 'red')]:
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_tr, y_tr)
    yp = model.predict(X_plot * 2)  # X_plot 是0-10，乘2变成0-20
    ax2.plot(X_plot*2, yp, label=label, color=color, lw=2, alpha=0.8)

ax2.scatter(X_tr[::5], y_tr[::5], alpha=0.4, s=20, label='训练集（采样）')
ax2.scatter(X_te[::5], y_te[::5], alpha=0.6, s=25, color='red', label='测试集（采样）')
ax2.set_xlabel('X'); ax2.set_ylabel('y')
ax2.set_title(f'泛化最优({best_deg}次) vs 过拟合({worst_deg}次)', fontsize=14)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day31/sample_out_validation.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'训练集上"最优"参数: {worst_deg}次多项式 → 测试MSE: {test_mse[worst_deg-1]:.1f}')
print(f'测试集上真正最优: {best_deg}次多项式 → 测试MSE: {test_mse[best_deg-1]:.1f}')
print(f'\n⚠️ 教训：在训练集上疯狂调参 = 在测试集上撞墙！')

## Part 4: ML策略实战 — 用模型生成交易信号

构建特征（技术指标）→ 训练RandomForest → 预测涨跌 → 回测验证

⚠️ 采用时序交叉验证（TimeSeriesSplit），不能打乱顺序！

In [ ]:
import akshare as ak
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

# ===== 1. 获取真实数据 =====
# 用新浪接口（东方财富偶尔抽风，新浪更稳）
df_raw = ak.stock_zh_a_daily(symbol='sh600519', start_date='20200101', 
                              end_date='20241231', adjust='qfq')
df_raw['date'] = pd.to_datetime(df_raw['date'])
df = df_raw.set_index('date')[['open', 'high', 'low', 'close', 'volume']]
df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
df = df.sort_index()

print(f'数据: {df.shape[0]} 天, {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")}')
print(f'标的: 贵州茅台(600519)')
df.tail(3)

### 4.1 构建特征 + 标签

- 用技术指标做特征
- 标签: 未来5天涨跌（未来收盘 > 今天收盘 = 1，否则 = 0）

In [5]:
# ===== 2. 特征工程 =====
def build_features(df):
    """构建技术指标特征矩阵"""
    data = df.copy()
    
    # 收益率特征
    data['ret_1d'] = data['Close'].pct_change(1)
    data['ret_5d'] = data['Close'].pct_change(5)
    data['ret_20d'] = data['Close'].pct_change(20)
    
    # 波动率
    data['vol_5d'] = data['ret_1d'].rolling(5).std()
    data['vol_20d'] = data['ret_1d'].rolling(20).std()
    
    # 均线偏离
    for window in [5, 10, 20, 60]:
        ma = data['Close'].rolling(window).mean()
        data[f'ma{window}_bias'] = (data['Close'] - ma) / ma  # 股价相对均线的偏离
    
    # RSI (手算14日)
    delta = data['Close'].diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    rs = gain / (loss + 1e-10)
    data['rsi_14'] = 100 - (100 / (1 + rs))
    
    # 成交量特征
    data['vol_ratio'] = data['Volume'] / data['Volume'].rolling(20).mean()
    
    # 标签：未来5天是否上涨
    data['target'] = (data['Close'].shift(-5) > data['Close']).astype(int)
    
    return data.dropna()

data = build_features(df)
feature_cols = [c for c in data.columns if c not in ['Open','High','Low','Close','Volume','target']]
print(f'特征数: {len(feature_cols)}: {feature_cols}')
print(f'标签分布: 涨={data["target"].sum()}天, 跌={len(data)-data["target"].sum()}天')
print(f'有效率: {len(data)}/{len(df)} 行')
data[feature_cols + ['target']].head()

特征数: 11: ['ret_1d', 'ret_5d', 'ret_20d', 'vol_5d', 'vol_20d', 'ma5_bias', 'ma10_bias', 'ma20_bias', 'ma60_bias', 'rsi_14', 'vol_ratio']
标签分布: 涨=578天, 跌=575天
有效率: 1153/1212 行


,ret_1d,ret_5d,ret_20d,vol_5d,vol_20d,ma5_bias,ma10_bias,ma20_bias,ma60_bias,rsi_14,vol_ratio,target
Date,,,,,,,,,,,,
2020-04-02,0.049778,0.099397,-0.028517,0.028149,0.035705,0.051898,0.084243,0.071854,0.068056,54.697250,0.902143,1
2020-04-03,-0.005882,0.078764,-0.017529,0.030408,0.035526,0.029983,0.064250,0.066563,0.061565,61.697683,0.552819,1
2020-04-07,0.022895,0.108214,0.053747,0.028312,0.034153,0.032324,0.070382,0.087955,0.084097,68.560716,0.870410,1
2020-04-08,-0.000211,0.057248,0.004193,0.024854,0.032395,0.020697,0.057121,0.087479,0.082093,77.850773,0.501837,1
2020-04-09,-0.001399,0.065779,-0.000022,0.023393,0.032395,0.006604,0.046015,0.085959,0.079196,81.042256,0.479005,1


### 4.2 时序交叉验证 + 训练

⚠️ 金融时序数据必须用 `TimeSeriesSplit`，不能用普通的 KFold！

In [6]:
# ===== 3. 时序划分 + 训练 =====
X = data[feature_cols].values
y = data['target'].values

# 前70%训练，后30%测试
split_idx = int(len(X) * 0.7)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# 在训练集上做时序交叉验证
tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train[tr_idx], y_train[tr_idx])
    score = model.score(X_train[val_idx], y_train[val_idx])
    cv_scores.append(score)
    print(f'Fold {fold+1}: val acc = {score:.4f}')

print(f'\nCV平均准确率: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')

# 最终模型：全量训练集 → 测试集评估
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f'\n训练集准确率: {accuracy_score(y_train, y_pred_train):.4f}')
print(f'测试集准确率: {accuracy_score(y_test, y_pred_test):.4f}')
print(f'\n⚠️ 训练-测试差距: {(accuracy_score(y_train, y_pred_train) - accuracy_score(y_test, y_pred_test)):.4f}')
print('差距 > 5% → 可能过拟合！')

Fold 1: val acc = 0.5970
Fold 2: val acc = 0.4328
Fold 3: val acc = 0.5299
Fold 4: val acc = 0.3731
Fold 5: val acc = 0.5149

CV平均准确率: 0.4896 ± 0.0782

训练集准确率: 0.8017
测试集准确率: 0.4653

⚠️ 训练-测试差距: 0.3364
差距 > 5% → 可能过拟合！


### 4.3 特征重要性 + 简易回测

看哪些特征对预测最重要，以及如果按信号交易能赚多少。

In [ ]:
# ===== 4. 特征重要性 =====
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(importance['feature'], importance['importance'])
ax.set_xlabel('重要性'); ax.set_title('RandomForest 特征重要性')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day31/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()
print(importance.to_string(index=False))

In [ ]:
# ===== 5. 简易回测 =====
# 按预测信号在测试集上模拟交易
test_data = data.iloc[split_idx:].copy()
test_data['signal'] = y_pred_test  # 1=看涨, 0=看跌

# 策略：信号=1时持有，信号=0时空仓
test_data['strategy_ret'] = test_data['ret_1d'] * test_data['signal'].shift(1)
test_data['buy_hold_ret'] = test_data['ret_1d']  # 基准：一直持有

# 累计收益曲线
test_data['strategy_cum'] = (1 + test_data['strategy_ret']).cumprod()
test_data['buyhold_cum'] = (1 + test_data['buy_hold_ret']).cumprod()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test_data.index, test_data['strategy_cum'], label='ML策略', lw=2)
ax.plot(test_data.index, test_data['buyhold_cum'], label='买入持有（基准）', alpha=0.7)
ax.set_title(f'贵州茅台(600519) ML策略 vs 买入持有（测试集）', fontsize=14)
ax.set_ylabel('累计收益'); ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/Users/macbookpro/Desktop/股/day31/backtest_result.png', dpi=100, bbox_inches='tight')
plt.show()

# 绩效指标
def calc_metrics(returns):
    sharpe = returns.mean() / (returns.std() + 1e-10) * np.sqrt(252)
    cum_ret = (1 + returns).prod() - 1
    max_dd = (1 + returns).cumprod().div((1 + returns).cumprod().cummax()) - 1
    return {'累计收益': f'{cum_ret:.2%}', '夏普比': f'{sharpe:.2f}', '最大回撤': f'{max_dd.min():.2%}'}

print('=== 绩效对比 ===')
print('ML策略:', calc_metrics(test_data['strategy_ret'].dropna()))
print('买入持有:', calc_metrics(test_data['buy_hold_ret'].dropna()))
print(f'\n交易次数: {((test_data["signal"].diff() != 0).sum())} 次')
print(f'持仓天数占比: {test_data["signal"].mean():.1%}')
print('\n⚠️ 现实提醒：这个回测没扣交易成本，实际表现会差一些')

## Part 5: Auquan 收尾总结

回顾 21 个 Notebook 的核心收获和待深入方向。

### 已掌握清单

| 主题 | 核心概念 | 掌握度 |
|------|---------|:-----:|
| 平稳性 | ADF检验、差分定阶 | L3 |
| 协整 | 配对交易基础、Engle-Granger | L2 |
| 过拟合 | 样本外验证、参数简约原则 | L3 |
| 伪回归 | 非平稳序列回归的危险 | L2 |
| ML框架 | 六步法：数据→特征→划分→训练→评估→回测 | L2 |
| ARIMA | AR/MA/ARMA 手算、AIC网格搜索 | L3 |
| GARCH | 波动率聚集、条件波动率 | L2 |
| ML策略 | RF信号生成、TimeSeriesSplit | L3 |

### 待深入方向（第2个月）
- 多因子策略组合
- XGBoost/LightGBM 替代 RandomForest
- SHAP 特征解释
- 更完善的回测框架（扣成本、滑点、仓位管理）